In [13]:
!pip install ultralytics
import os
import shutil
import zipfile
import yaml
from ultralytics import YOLO
from google.colab import files

In [14]:
# --- Configuration ---
zip_path = '/content/dataset.zip'       # Your uploaded zip
temp_path = '/content/temp_extract'     # Temporary holding area
final_path = '/content/dataset'         # The clean destination

# 1. Clean up previous runs
if os.path.exists(final_path):
    shutil.rmtree(final_path)
if os.path.exists(temp_path):
    shutil.rmtree(temp_path)

# 2. Extract to a temporary folder
print("Extracting zip...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_path)

# 3. Find the TRUE root folder (the one containing data.yaml)
data_root = None
for root, dirs, files in os.walk(temp_path):
    if 'data.yaml' in files:
        data_root = root
        break

if data_root is None:
    raise FileNotFoundError("Could not find data.yaml! Please check your zip file.")

# 4. Move correct files to /content/dataset
print(f"Found data at: {data_root}")
shutil.move(data_root, final_path)

# 5. Cleanup
if os.path.exists(temp_path):
    shutil.rmtree(temp_path)

print(f"Success! Dataset is ready at: {final_path}")

Extracting zip...
Found data at: /content/temp_extract/dataset
Success! Dataset is ready at: /content/dataset


In [15]:
yaml_path = '/content/dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to absolute Colab paths
data['train'] = '/content/dataset/images/train'
data['val'] = '/content/dataset/images/val'
data['test'] = '/content/dataset/images/test'

# Ensure correct classes
data['nc'] = 3
data['names'] = ['ABB', 'Kapital Bank', 'Pasha Bank']

# Save changes
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"Updated YAML file at: {yaml_path}")
print("Classes set to:", data['names'])

Updated YAML file at: /content/dataset/data.yaml
Classes set to: ['ABB', 'Kapital Bank', 'Pasha Bank']


In [16]:
# Initialize YOLOv5 Small
model_s = YOLO('yolov5su.pt')

print("\n--- Starting Training: YOLOv5s (Baseline) ---")
results_s = model_s.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    patience=15,
    imgsz=640,
    batch=16,
    project='/content/runs/detect',
    name='yolov5s_baseline',
    exist_ok=True,
    verbose=True,

    # Minimal Augmentation
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)
print("Training Finished!")


--- Starting Training: YOLOv5s (Baseline) ---
Ultralytics 8.4.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5su.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov5s_baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [17]:
# Initialize YOLOv5 Medium
model_m = YOLO('yolov5mu.pt')

print("\n--- Starting Training: YOLOv5m (With Augmentation) ---")
results = model_m.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    patience=15,            # Wait longer before stopping
    imgsz=640,
    batch=16,
    project='/content/runs/detect',
    name='bank_logo_aug',   # New name for this run
    exist_ok=True,
    verbose=True,

    # --- AUGMENTATION SETTINGS ---
    degrees=15.0,      # Rotate +/- 15 degrees
    translate=0.1,     # Shift image +/- 10%
    scale=0.5,         # Zoom +/- 50%
    shear=2.0,         # Shear +/- 2 degrees
    fliplr=0.5,        # Flip Left-Right 50% of time
    mosaic=1.0,        # Use Mosaic (4-image stitch)
    mixup=0.1,         # Mixup (blend 2 images) - set low for logos
    hsv_h=0.015,       # Adjust Hue slightly
    hsv_s=0.7,         # Adjust Saturation
    hsv_v=0.4          # Adjust Brightness
)

print("Training Finished!")


--- Starting Training: YOLOv5m (With Augmentation) ---
Ultralytics 8.4.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov5mu.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bank_logo_aug, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

In [22]:
from google.colab import files

# --- Define paths to the best weights ---
path_s = '/content/runs/detect/yolov5s_baseline/weights/best.pt'
path_m = '/content/runs/detect/bank_logo_aug/weights/best.pt'

print("--- Training Complete ---")

# Download YOLOv5s Baseline with custom name
if os.path.exists(path_s):
    new_name_s = 'yolov5s_baseline.pt'
    shutil.copy(path_s, f'/content/{new_name_s}')  # Copy to /content with new name
    print(f"Downloading YOLOv5s Baseline as: {new_name_s}")
    files.download(f'/content/{new_name_s}')

# Download YOLOv5m Augmented with custom name
if os.path.exists(path_m):
    new_name_m = 'yolov5m_augmented.pt'
    shutil.copy(path_m, f'/content/{new_name_m}')  # Copy to /content with new name
    print(f"Downloading YOLOv5m Augmented as: {new_name_m}")
    files.download(f'/content/{new_name_m}')

--- Training Complete ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>